In [ ]:
import re
import os
import sys
import json
sys.path.append('../scripts')

from utils import ITEM_PATTERNS, ITEM_NAMES
from pinecone import Pinecone
from time import sleep

In [ ]:
with open('../cfg.json', 'r') as f:
    config = json.load(f)
read_folder_path = config['processed_data_folder']

In [ ]:
# create meaningful chunks from 10-K filings
def chunk_10k(text: str, ticker: str, fiscal_year: str, chunk_size: int = 400, overlap: int = 50) -> list[dict]:
    chunks = []
    doc_len = len(text)
    true_item1_start = None

    print(f"""Processing {ticker} ...""")
    
    # find all item1 and item1a positions
    item1_matches = list(re.finditer(ITEM_PATTERNS["item_1"], text, re.IGNORECASE))
    item1a_matches = list(re.finditer(ITEM_PATTERNS["item_1a"], text, re.IGNORECASE))
    
    if not item1_matches or len(item1_matches) == 0:
        raise Exception(f"No ITEM 1 found for {ticker}")
    if not item1a_matches or len(item1a_matches) == 0:
        raise Exception(f"No ITEM 1A found for {ticker}")
    
    print("Item1 Matches", item1_matches)
    print("Item1a Matches", item1a_matches)
    
    # figure out where is the true item1 start (not table of contents)
    for i in range(len(item1_matches)):
        item1_pos = item1_matches[i].start()
        item1a_pos = item1a_matches[i].start()            

        if item1a_pos > item1_pos:
            section_size = item1a_pos - item1_pos
        else:
            continue
        
        if section_size > 5000:
            true_item1_start = item1_pos
            break

    # fallback if logic fails
    if true_item1_start is None:
        print("!True Item 1 could not be found. Defaulting to the first one.")
        true_item1_start = item1_matches[0]

    print("True Item1 Start", true_item1_start)
    print("Length of the doc", doc_len)
       
    # separate sections
    item_positions = []
    for key, pattern in ITEM_PATTERNS.items():
        for match in re.finditer(pattern, text, re.IGNORECASE):
            if match.start() >= true_item1_start:
                item_positions.append((key, match.start()))
                break  # first occurrence per item
    
    item_positions.sort(key=lambda x: x[1])
    print(item_positions)
    
    # start actual chunking
    for idx, (key, start_pos) in enumerate(item_positions):
        end_pos = item_positions[idx + 1][1] if idx + 1 < len(item_positions) else doc_len
        section_text = text[start_pos:end_pos].strip()
        
        # split section into chunks with overlap
        tokens = section_text.split()
        chunk_num = 0

        for chunk_start in range(0, len(tokens), chunk_size - overlap):
            chunk_end = chunk_start + chunk_size
            chunk = tokens[chunk_start:chunk_end]
            
            chunk_text = ITEM_NAMES[key] + "\n" + " ".join(chunk)
            
            chunks.append({
                "_id": f"{ticker}_{fiscal_year}_chunk{chunk_num}",
                "chunk_text": chunk_text,
                "ticker": ticker,
                "fiscal_year": fiscal_year,
                "item": key,
                "chunk_size": len(chunk_text)
            })
            
            chunk_num += 1    
    return chunks

In [ ]:
all_chunks = []

# bring all the chunks together under one list
for file_name in os.listdir(read_folder_path):
    if file_name.endswith('.txt'):

        file_path = os.path.join(read_folder_path, file_name)
        ticker = file_name.split('_')[0]
        filing_year = file_name.split('-')[1]
        
        print(f"Processing {ticker} | FY {filing_year}")        

        with open(file_path, 'r') as f:
            text = f.read()
            all_chunks += (chunk_10k(text, ticker, filing_year))


Processing AAPL | FY 25
Processing AAPL ...
[<re.Match object; span=(19280, 19296), match='Item 1. Business'>, <re.Match object; span=(22564, 22583), match='Item 1.    Business'>]
[<re.Match object; span=(19299, 19320), match='Item 1A. Risk Factors'>, <re.Match object; span=(38636, 38660), match='Item 1A.    Risk Factors'>]
22564
222577
[('item_1', 22564), ('item_1a', 38636), ('item_1b', 106688), ('item_2', 109450), ('item_3', 109970), ('item_4', 115374), ('item_5', 115466), ('item_6', 118177), ('item_7', 118232), ('item_7a', 136419), ('item_8', 139460), ('item_9', 201722), ('item_9a', 201824), ('item_9b', 206328), ('item_10', 207006), ('item_11', 207407), ('item_12', 207567), ('item_13', 207799), ('item_14', 208010), ('item_15', 208227)]
7
Processing AMZN | FY 24
Processing AMZN ...
[<re.Match object; span=(33195, 33211), match='Item 1. Business'>, <re.Match object; span=(34518, 34534), match='Item 1. Business'>]
[<re.Match object; span=(33214, 33235), match='Item 1A. Risk Factors'>, 

In [ ]:
pc = Pinecone(api_key=config['pinecone_api_key'])

# Define index names
dense_index_name = "10k-dense"
sparse_index_name = "10k-sparse"

# create dense index
if not pc.has_index(dense_index_name):
    pc.create_index_for_model(
        name=dense_index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"llama-text-embed-v2",
            "field_map":{"text": "chunk_text"}
        }
    )

# create sparse index
if not pc.has_index(sparse_index_name):
    pc.create_index_for_model(
        name=sparse_index_name,
        cloud="aws",
        region="us-east-1",
        embed={
            "model":"pinecone-sparse-english-v0",
            "field_map":{"text": "chunk_text"}
        }
    )

dense_index = pc.Index(dense_index_name)
sparse_index = pc.Index(sparse_index_name)

# upsert in batches of maximum 96 records with 13 seconds sleep between batches
batch_size = 96
total_chunks = len(all_chunks)

print(f"Upserting {total_chunks} records in batches of {batch_size}...")

for i in range(0, total_chunks, batch_size):
    batch = all_chunks[i:i + batch_size]
    batch_num = (i // batch_size) + 1
    
    print(f"  Batch {batch_num}: Upserting {len(batch)} records...")
    dense_index.upsert_records("namespace", batch)
    sparse_index.upsert_records("namespace", batch)
    sleep(13)

print("✓ All records upserted successfully!")

Upserting 1322 records in batches of 96...
  Batch 1: Upserting 96 records...
  Batch 2: Upserting 96 records...
  Batch 3: Upserting 96 records...
  Batch 4: Upserting 96 records...
  Batch 5: Upserting 96 records...
  Batch 6: Upserting 96 records...
  Batch 7: Upserting 96 records...
  Batch 8: Upserting 96 records...
  Batch 9: Upserting 96 records...
  Batch 10: Upserting 96 records...
  Batch 11: Upserting 96 records...
  Batch 12: Upserting 96 records...
  Batch 13: Upserting 96 records...
  Batch 14: Upserting 74 records...
✓ All records upserted successfully!


In [5]:
sleep(60)

In [6]:
print(dense_index.describe_index_stats())

{'dimension': 1024,
 'index_fullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'namespace': {'vector_count': 459}},
 'total_vector_count': 459,
 'vector_type': 'dense'}


In [7]:
dense_index.search(
    namespace="namespace",
    query={
        "top_k": 5,
        "inputs": {
            "text": "What was Apple's revenue from the iPad product line?"
        }
    }
)

{'result': {'hits': [{'_id': 'AAPL_25_chunk7',
                      '_score': 0.4369759261608124,
                      'fields': {'chunk_size': 2556.0,
                                 'chunk_text': 'Item 8 Financial Statements '
                                               'and Supplementary Data\n'
                                               'previously deferred, for 2025, '
                                               '2024 and 2023 (in millions): '
                                               '2025 | 2024 | 2023 iPhone: '
                                               '$209,586 | $201,183 | $200,583 '
                                               'Mac: 33,708 | 29,984 | 29,357 '
                                               'iPad: 28,023 | 26,694 | 28,300 '
                                               'Wearables, Home and '
                                               'Accessories: 35,686 | 37,005 | '
                                               '39,845 Services 